# SAC Colab Runner (bonus: off-policy expert)

Trains SAC experts on a Colab **GPU** runtime (SAC is update-bound, so a GPU helps, unlike the CPU-bound PPO). Models persist to Google Drive so a disconnect does not lose progress. The rest of the pipeline (PPO, BC, DAgger) is reproduced from the README run scripts, not here.

**First:** Runtime -> Change runtime type -> **GPU**. Then run the cells top to bottom. No numpy downgrade or restart is needed for SAC.

## 1. Clone the code

In [ ]:
REPO = 'https://github.com/em-ech/rl-ppo-imitation-learning.git'
BRANCH = 'extended-e1-e2-and-sac-bonus'  # set to 'main' once merged
CODE_DIR = '/content/GroupProject'
import os, sys, shutil
os.chdir('/content')  # never stand inside the dir we delete
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)
!git clone -q --branch {BRANCH} {REPO} {CODE_DIR}
sys.path.insert(0, CODE_DIR); os.chdir(CODE_DIR)
assert os.path.exists('train_sac.py') and os.path.exists('src/sac.py'), 'SAC code missing'
print('code ready in', CODE_DIR)

## 2. Install dependencies (lean, no restart)

Only stable-baselines3 and gymnasium[mujoco] are needed for SAC, both numpy-2 compatible, so Colab's preinstalled numpy is left alone and no runtime restart is required.

_If a MuJoCo import error appears, fall back to the pinned stack: `!pip -q install -r requirements.txt`, then Runtime -> Restart session, then re-run cells 1-3._

In [ ]:
!pip -q install "stable-baselines3>=2.4.0" "gymnasium[mujoco]>=1.0.0"
print('install done')

## 3. Mount Drive, configure persistence, check GPU

Pointing `PROJECT_DATA_ROOT` at Drive sends `models/` and `logs/` to Drive, so checkpoints (every 100k steps) and `best_model.zip` survive a disconnect. The `!python` training calls below inherit this env var.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch
DRIVE_ROOT = '/content/drive/MyDrive/rl_project'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ['PROJECT_DATA_ROOT'] = DRIVE_ROOT
print('GPU available:', torch.cuda.is_available())
print('persisting models/logs to', DRIVE_ROOT)
import gymnasium as gym
for e in ['Ant-v4', 'HalfCheetah-v4']:
    gym.make(e).reset(seed=0); print('env ok:', e)

## 4. Train SAC - Ant-v4

In-brief comparison against the PPO expert (target 4000; SAC realistically ~5000-7000). ~1-2h on a GPU. If it disconnects, re-run the resume cell.

In [ ]:
!cd /content/GroupProject && python train_sac.py Ant-v4 3000000 tuned_ant

In [ ]:
# Resume Ant after a disconnect (continues from the last Drive checkpoint):
# !cd /content/GroupProject && python train_sac.py Ant-v4 3000000 tuned_ant resume

## 5. Train SAC - HalfCheetah-v4

Off-brief, the clean path past 8000 (SAC typically ~9000-11000). Saves to a separate Drive folder, so it does not collide with the Ant run.

In [ ]:
!cd /content/GroupProject && python train_sac.py HalfCheetah-v4 3000000 tuned_halfcheetah

In [ ]:
# Resume HalfCheetah after a disconnect:
# !cd /content/GroupProject && python train_sac.py HalfCheetah-v4 3000000 tuned_halfcheetah resume

## Recover after a disconnect or session restart

The code is on GitHub and the trained artifacts are on Drive under `MyDrive/rl_project/models/sac_expert_<env>/` (`best_model.zip` plus `sac_checkpoint_*_steps.zip`). After a restart, re-run cells 1-3 to re-clone, re-install, and re-mount, then run the matching resume cell. `best_model` is saved by eval return, so a restart never loses your best policy; only training since the last 100k checkpoint and the replay buffer are redone.

**A restart only loses progress that was not on Drive.** Always run cell 3 (which sets `PROJECT_DATA_ROOT`) before training.